In [0]:
#Read Bronze and Silver tables
bronze_df = spark.table("workspace.default.bronze_hhs_healthcare_breaches")
silver_df = spark.table("workspace.default.silver_hhs_healthcare_breaches")

print("Bronze records:", bronze_df.count())
print("Silver records:", silver_df.count())

Bronze records: 725
Silver records: 725


In [0]:
#Check null values in important columns
from pyspark.sql.functions import col, sum as spark_sum, when

important_columns = [
    "covered_entity_name",
    "state",
    "covered_entity_type",
    "individuals_affected",
    "breach_submission_date",
    "type_of_breach",
    "location_of_breached_information",
    "risk_level"
]

null_check_df = silver_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c + "_nulls")
    for c in important_columns
])

display(null_check_df)

covered_entity_name_nulls,state_nulls,covered_entity_type_nulls,individuals_affected_nulls,breach_submission_date_nulls,type_of_breach_nulls,location_of_breached_information_nulls,risk_level_nulls
0,3,0,0,0,0,0,0


In [0]:
#Checking which 3 records have missing state
from pyspark.sql.functions import col

display(
    silver_df
    .filter(col("state").isNull())
    .select(
        "covered_entity_name",
        "state",
        "covered_entity_type",
        "individuals_affected",
        "breach_submission_date",
        "type_of_breach",
        "location_of_breached_information",
        "risk_level"
    )
)

covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,risk_level
Hospital Caribbean Medical Center,null,Healthcare Provider,92000,2026-04-06,Hacking/IT Incident,Network Server,High
CORPORACION DE SERVICIOS MEDICOS PRIMARIOS Y PREVENCION DE HATILLO,null,Healthcare Provider,24236,2026-03-26,Hacking/IT Incident,Network Server,High
Schneider Regional Medical Center,null,Healthcare Provider,1570,2024-10-07,Hacking/IT Incident,Network Server,High


In [0]:
from pyspark.sql.functions import col, trim, when

silver_df = silver_df.withColumn(
    "state",
    when(
        col("state").isNull() | (trim(col("state")) == ""),
        "UNKNOWN"
    ).otherwise(col("state"))
)

display(silver_df)

covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,business_associate_present,web_description,ingestion_time,source_file_path,breach_year,breach_month,risk_level
University of Michigan/Michigan Medicine,MI,Healthcare Provider,551,2026-05-01,Unauthorized Access/Disclosure,Electronic Medical Record,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,5,Medium
Mt. Spokane Pediatrics,WA,Healthcare Provider,32021,2026-04-30,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High
"Ouster, Inc.",CA,Health Plan,574,2026-04-30,Unauthorized Access/Disclosure,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,Medium
Northwoods Surgery Center,MN,Healthcare Provider,5385,2026-04-29,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High
Tri-Cities Gastroenterology,TN,Healthcare Provider,67115,2026-04-29,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High
Belmont Aesthetic and Reconstructive Plastic Surgery,VA,Healthcare Provider,528,2026-04-23,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High
Interim HealthCare of Lubbock,TX,Healthcare Provider,2071,2026-04-23,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High
Interim HealthCare of Amarillo,TX,Healthcare Provider,666,2026-04-23,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High
Liberty Bankers Life Ins. Co.,TX,Health Plan,20202,2026-04-22,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High
"Spokane Digestive Disease Center, P.S.",WA,Healthcare Provider,501,2026-04-20,Hacking/IT Incident,Email,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High


In [0]:
#Add a data quality flag
from pyspark.sql.functions import lit

silver_df = silver_df.withColumn(
    "state_quality_flag",
    when(col("state") == "UNKNOWN", "Missing in source")
    .otherwise("Available")
)

display(silver_df)

covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,business_associate_present,web_description,ingestion_time,source_file_path,breach_year,breach_month,risk_level,state_quality_flag
University of Michigan/Michigan Medicine,MI,Healthcare Provider,551,2026-05-01,Unauthorized Access/Disclosure,Electronic Medical Record,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,5,Medium,Available
Mt. Spokane Pediatrics,WA,Healthcare Provider,32021,2026-04-30,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
"Ouster, Inc.",CA,Health Plan,574,2026-04-30,Unauthorized Access/Disclosure,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,Medium,Available
Northwoods Surgery Center,MN,Healthcare Provider,5385,2026-04-29,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
Tri-Cities Gastroenterology,TN,Healthcare Provider,67115,2026-04-29,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
Belmont Aesthetic and Reconstructive Plastic Surgery,VA,Healthcare Provider,528,2026-04-23,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
Interim HealthCare of Lubbock,TX,Healthcare Provider,2071,2026-04-23,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
Interim HealthCare of Amarillo,TX,Healthcare Provider,666,2026-04-23,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
Liberty Bankers Life Ins. Co.,TX,Health Plan,20202,2026-04-22,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
"Spokane Digestive Disease Center, P.S.",WA,Healthcare Provider,501,2026-04-20,Hacking/IT Incident,Email,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available


In [0]:
display(
    silver_df
    .filter(col("state").isNull())
)

covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,business_associate_present,web_description,ingestion_time,source_file_path,breach_year,breach_month,risk_level,state_quality_flag


In [0]:
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.silver_hhs_healthcare_breaches"
)

In [0]:
#record count validation

bronze_count = bronze_df.count()
silver_count = silver_df.count()

record_count_check = spark.createDataFrame(
    [
        ("bronze_hhs_healthcare_breaches", bronze_count),
        ("silver_hhs_healthcare_breaches", silver_count)
    ],
    ["table_name", "record_count"]
)

display(record_count_check)

table_name,record_count
bronze_hhs_healthcare_breaches,725
silver_hhs_healthcare_breaches,725


In [0]:
record_count_check.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.dq_record_count_check"
)

In [0]:
from pyspark.sql.functions import col, trim

important_columns = [
    "covered_entity_name",
    "state",
    "covered_entity_type",
    "individuals_affected",
    "breach_submission_date",
    "type_of_breach",
    "location_of_breached_information",
    "risk_level"
]

null_results = []

for c in important_columns:
    missing_count = silver_df.filter(
        col(c).isNull() | (trim(col(c).cast("string")) == "")
    ).count()
    null_results.append((c, missing_count))

null_check_df = spark.createDataFrame(
    null_results,
    ["column_name", "missing_count"]
)

display(null_check_df)

column_name,missing_count
covered_entity_name,0
state,0
covered_entity_type,0
individuals_affected,0
breach_submission_date,0
type_of_breach,0
location_of_breached_information,0
risk_level,0


In [0]:
null_check_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.dq_null_check"
)

In [0]:
#duplicate records
from pyspark.sql.functions import count

duplicate_records_df = (
    silver_df
    .groupBy(
        "covered_entity_name",
        "state",
        "individuals_affected",
        "breach_submission_date",
        "type_of_breach"
    )
    .agg(count("*").alias("duplicate_count"))
    .filter(col("duplicate_count") > 1)
)

display(duplicate_records_df)

covered_entity_name,state,individuals_affected,breach_submission_date,type_of_breach,duplicate_count
McEwen & Associates,TX,500,2025-08-21,Hacking/IT Incident,3


In [0]:
duplicate_group_count = duplicate_records_df.count()
print("Duplicate record groups:", duplicate_group_count)

Duplicate record groups: 1


In [0]:
#inspect duplicate rows
duplicate_keys = [
    "covered_entity_name",
    "state",
    "individuals_affected",
    "breach_submission_date",
    "type_of_breach"
]

display(
    silver_df
    .filter(
        (col("covered_entity_name") == "McEwen & Associates") &
        (col("state") == "TX") &
        (col("individuals_affected") == 500) &
        (col("breach_submission_date") == "2025-08-21") &
        (col("type_of_breach") == "Hacking/IT Incident")
    )
)

covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,business_associate_present,web_description,ingestion_time,source_file_path,breach_year,breach_month,risk_level,state_quality_flag
McEwen & Associates,TX,Business Associate,500,2025-08-21,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2025,8,High,Available
McEwen & Associates,TX,Business Associate,500,2025-08-21,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2025,8,High,Available
McEwen & Associates,TX,Business Associate,500,2025-08-21,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2025,8,High,Available


In [0]:
#removing duplicate records
deduped_silver_df = silver_df.dropDuplicates([
    "covered_entity_name",
    "state",
    "individuals_affected",
    "breach_submission_date",
    "type_of_breach"
])

print("Before deduplication:", silver_df.count())
print("After deduplication:", deduped_silver_df.count())
print("Duplicate rows removed:", silver_df.count() - deduped_silver_df.count())

display(deduped_silver_df)

Before deduplication: 725
After deduplication: 723
Duplicate rows removed: 2


covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,business_associate_present,web_description,ingestion_time,source_file_path,breach_year,breach_month,risk_level,state_quality_flag
University of Michigan/Michigan Medicine,MI,Healthcare Provider,551,2026-05-01,Unauthorized Access/Disclosure,Electronic Medical Record,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,5,Medium,Available
Mt. Spokane Pediatrics,WA,Healthcare Provider,32021,2026-04-30,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
"Ouster, Inc.",CA,Health Plan,574,2026-04-30,Unauthorized Access/Disclosure,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,Medium,Available
Northwoods Surgery Center,MN,Healthcare Provider,5385,2026-04-29,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
Tri-Cities Gastroenterology,TN,Healthcare Provider,67115,2026-04-29,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
Belmont Aesthetic and Reconstructive Plastic Surgery,VA,Healthcare Provider,528,2026-04-23,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
Interim HealthCare of Lubbock,TX,Healthcare Provider,2071,2026-04-23,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
Interim HealthCare of Amarillo,TX,Healthcare Provider,666,2026-04-23,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
Liberty Bankers Life Ins. Co.,TX,Health Plan,20202,2026-04-22,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available
"Spokane Digestive Disease Center, P.S.",WA,Healthcare Provider,501,2026-04-20,Hacking/IT Incident,Email,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv,2026,4,High,Available


In [0]:
display(
    silver_df
    .filter(col("state").isNull())
)

covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,business_associate_present,web_description,ingestion_time,source_file_path,breach_year,breach_month,risk_level,state_quality_flag


In [0]:
deduped_silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.silver_hhs_healthcare_breaches"
)

In [0]:
silver_df = spark.table("workspace.default.silver_hhs_healthcare_breaches")
print("Final Silver record count:", silver_df.count())

Final Silver record count: 723


In [0]:
#Invalid individuals_affected check
#Healthcare breach records should not have zero or negative affected individuals.
invalid_individuals_df = silver_df.filter(
    (col("individuals_affected").isNull()) |
    (col("individuals_affected") <= 0)
)

display(invalid_individuals_df)

covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,business_associate_present,web_description,ingestion_time,source_file_path,breach_year,breach_month,risk_level,state_quality_flag


In [0]:
print("Invalid individuals_affected records:", invalid_individuals_df.count())

Invalid individuals_affected records: 0


In [0]:
invalid_individuals_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.dq_invalid_individuals_affected"
)

In [0]:
#data validation check
invalid_date_df = silver_df.filter(
    col("breach_submission_date").isNull()
)

display(invalid_date_df)

covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,business_associate_present,web_description,ingestion_time,source_file_path,breach_year,breach_month,risk_level,state_quality_flag


In [0]:
print("Invalid or missing breach dates:", invalid_date_df.count())

Invalid or missing breach dates: 0


In [0]:
invalid_date_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.dq_invalid_breach_dates"
)

In [0]:
#Risk level validation
risk_level_check_df = (
    silver_df
    .groupBy("risk_level")
    .count()
    .orderBy("count", ascending=False)
)

display(risk_level_check_df)

risk_level,count
High,631
Medium,85
Low,7


In [0]:
#check invalid risk levels:
invalid_risk_df = silver_df.filter(
    ~col("risk_level").isin("High", "Medium", "Low")
)

display(invalid_risk_df)
print("Invalid risk level records:", invalid_risk_df.count())

covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,business_associate_present,web_description,ingestion_time,source_file_path,breach_year,breach_month,risk_level,state_quality_flag


Invalid risk level records: 0


In [0]:
risk_level_check_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.dq_risk_level_distribution"
)

In [0]:
#Final data quality summary table
dq_summary_data = [
    ("Bronze Record Count", bronze_count),
    ("Silver Record Count", silver_count),
    ("Record Count Difference", bronze_count - silver_count),
    ("Duplicate Record Groups", duplicate_group_count),
    ("Invalid Individuals Affected Records", invalid_individuals_df.count()),
    ("Invalid Breach Date Records", invalid_date_df.count()),
    ("Invalid Risk Level Records", invalid_risk_df.count())
]

dq_summary_df = spark.createDataFrame(
    dq_summary_data,
    ["check_name", "check_result"]
)

display(dq_summary_df)


check_name,check_result
Bronze Record Count,725
Silver Record Count,725
Record Count Difference,0
Duplicate Record Groups,1
Invalid Individuals Affected Records,0
Invalid Breach Date Records,0
Invalid Risk Level Records,0


In [0]:
dq_summary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.dq_healthcare_breach_summary"
)

In [0]:
#Validated Bronze and Silver record counts
#Checked missing values in critical healthcare breach fields
#Identified and handled missing state values safely
#Checked duplicate breach records
#Validated affected-individual counts
#Checked missing breach dates
#Validated High/Medium/Low risk-level classification
#Created Delta-based data quality summary tables